# GraphRAG 与混合检索 — 第四节
## GraphRAG & Hybrid Retrieval — Part 4

**核心主题 / Core Topic：**

> 端到端 GraphRAG 流水线 — LLM 抽取实体 → Neo4j 存图 → 图谱问答 + 向量检索 → LLM 合成答案  
> End-to-end GraphRAG pipeline: Entity extraction → Neo4j storage → Graph Q&A + Vector Q&A → LLM synthesis

---

### 完整 Pipeline 架构

```
原始文档
   │
   ├──► LLMGraphTransformer ──► Neo4j 知识图谱
   │                                  │
   └──► SentenceTransformer  ──► Chroma 向量库
                                       │
用户问题 ──► GraphCypherQAChain ────────┤
         └──► Vector similarity search ┤
                                       ▼
                              LLM 综合两路答案
                                       │
                                  最终回答
```

### 五个核心步骤

| Step | 组件 | 功能 |
|------|------|------|
| 1 | LLM + Neo4j | 连接 LLM 与图数据库 |
| 2 | LLMGraphTransformer | NL 文档 → 图三元组 (实体, 关系, 实体) |
| 3 | Chroma + MiniLM | 构建向量检索库 |
| 4 | GraphCypherQAChain | NL 问题 → Cypher → 图查询 |
| 5 | Hybrid + LLM | 融合图答案 + 向量上下文，生成最终答案 |

---

**前置条件 / Prerequisites:**

> ⚠️ **Notebook 4 需要 Neo4j + APOC 插件**（LangChain 用 `apoc.meta.data()` 读 Schema，用 `apoc.merge.*` 写入图）

```bash
# 1. 启动 Neo4j（必须启用 APOC）
docker run -d --name neo4j-apoc \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  -e NEO4J_PLUGINS='["apoc"]' \
  -e NEO4J_dbms_security_procedures_unrestricted=apoc.* \
  -e NEO4J_dbms_security_procedures_allowlist=apoc.* \
  neo4j:latest

# 2. 设置环境变量（.env 文件）
# OPENAI_API_KEY=your_deepseek_key

pip install langchain langchain-openai langchain-community langchain-neo4j \
            chromadb sentence-transformers python-dotenv openai langchain-experimental \
            json-repair
```

**Neo4j Desktop 用户**：数据库 → Manage → Plugins → Install **APOC** → 重启数据库。

## step 1 — 导入依赖 / Import Dependencies

In [ ]:
from __future__ import annotations

import os
from typing import Any

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_experimental.graph_transformers import LLMGraphTransformer

print("✅ 依赖导入成功")

## step 2 — 环境配置 / Configuration

从 `.env` 文件加载密钥，支持 **DeepSeek API**（兼容 OpenAI 格式）。

`.env` 文件格式：
```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxx
NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=password
```

In [ ]:
load_dotenv()

NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"

if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY 未设置，请在 .env 文件中添加。")

print(f"Neo4j URI : {NEO4J_URI}")
print(f"LLM Base  : {DEEPSEEK_BASE_URL}")
print(f"API Key   : {'*' * 8}{OPENAI_API_KEY[-4:] if len(OPENAI_API_KEY) > 4 else '****'}")

## step 3 — 原始文档 / Raw Documents

这些文档将被 **LLMGraphTransformer** 解析为实体和关系三元组，
同时也会被向量化存入 **Chroma**。

LangChain 的 `Document` 对象包装纯文本，便于 Pipeline 处理。

In [ ]:
RAW_TEXTS: list[str] = [
    "Apple Inc. was founded by Steve Jobs in 1976. Tim Cook is the current CEO of Apple.",
    "Tim Cook studied Industrial Engineering at Auburn University located in Auburn, Alabama.",
    "Apple makes the iPhone 15 Pro Max and the MacBook Pro M3.",
    "Google was founded by Larry Page and Sergey Brin in 1998. Sundar Pichai is Google's CEO.",
    "Sundar Pichai attended Stanford University in Stanford, California.",
    "Google produces Google Search and the Pixel 8 smartphone.",
    "Microsoft was founded by Bill Gates in 1975. Satya Nadella became CEO in 2014.",
    "Microsoft develops Windows 11 and Azure Cloud services.",
]

# LangChain Document 包装
DOCUMENTS: list[Document] = [Document(page_content=text) for text in RAW_TEXTS]

print(f"共 {len(RAW_TEXTS)} 条文档:")
for i, text in enumerate(RAW_TEXTS):
    print(f"  [{i}] {text}")

## step 4 — 构建 LLM / Build LLM

使用 **DeepSeek** 作为后端 LLM（通过 OpenAI 兼容接口）。

- `model="deepseek-chat"`：DeepSeek 通用对话模型（推荐）
- `temperature=0`：确保实体抽取结果稳定可复现

> ⚠️ DeepSeek **不支持** OpenAI 的 `response_format` 结构化输出。  
> Step 6 中 `LLMGraphTransformer` 需设置 `ignore_tool_usage=True`，改用 JSON 文本解析。

In [ ]:
def build_llm() -> ChatOpenAI:
    """Build a ChatOpenAI-compatible LLM pointed at DeepSeek."""
    return ChatOpenAI(
        model="deepseek-v4-pro",
        api_key=OPENAI_API_KEY,
        base_url=DEEPSEEK_BASE_URL,
        temperature=0,
    )


llm = build_llm()
print("✅ LLM 初始化完成")
print(f"   Model: {llm.model_name}")

## step 5 — 连接 Neo4j / Connect to Neo4j

> ⚠️ **必须安装 APOC 插件**，否则 `Neo4jGraph` 初始化会失败。

LangChain 依赖 APOC 做两件事：
- `apoc.meta.data()` — 读取图 Schema（供 GraphCypherQAChain 生成 Cypher）
- `apoc.merge.*` — 写入实体和关系（Step 6 `add_graph_documents`）

**Neo4j Desktop 安装 APOC：**
1. 选中数据库 → **Manage** → **Plugins** → 安装 **APOC**
2. 打开 **Settings**，确认配置包含：
   ```
   dbms.security.procedures.unrestricted=apoc.*
   dbms.security.procedures.allowlist=apoc.*
   ```
3. **Restart** 数据库

**Docker 启动（含 APOC）：**
```bash
docker run -d --name neo4j-apoc \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  -e NEO4J_PLUGINS='["apoc"]' \
  -e NEO4J_dbms_security_procedures_unrestricted=apoc.* \
  -e NEO4J_dbms_security_procedures_allowlist=apoc.* \
  neo4j:latest
```

安装后可在 Neo4j Browser 验证：`RETURN apoc.version()`

In [ ]:
def build_neo4j_graph() -> Neo4jGraph:
    """Connect to Neo4j and verify APOC is available."""
    graph = Neo4jGraph(
        url=NEO4J_URI,
        username=NEO4J_USER,
        password=NEO4J_PASSWORD,
    )
    # Quick APOC sanity check
    version = graph.query("RETURN apoc.version() AS version")
    print(f"  APOC version: {version[0]['version']}")
    return graph


try:
    graph = build_neo4j_graph()
    print("✅ Neo4j 连接成功")
except ValueError as e:
    if "APOC" in str(e):
        raise RuntimeError(
            "Neo4j 未启用 APOC 插件。请按上方 Step 5 说明安装 APOC 并重启数据库，然后重新运行本 cell。"
        ) from e
    raise

## step 6 — LLM 实体抽取 → 存入图谱 / Entity Extraction & Graph Storage

**LLMGraphTransformer** 的工作原理：

```
输入文本："Tim Cook is the CEO of Apple."
         │
         ▼  LLM 解析
三元组：(Tim Cook) -[IS_CEO_OF]-> (Apple)
         │
         ▼  graph.add_graph_documents()
Neo4j 节点 + 边
```

参数说明：
- `ignore_tool_usage=True`：**DeepSeek 兼容模式**，不用 `response_format`，改走 JSON 文本解析
- `baseEntityLabel=True`：为所有节点添加通用 `__Entity__` 标签，便于统一查询
- `include_source=True`：保留原始文档引用，支持溯源

In [ ]:
def extract_and_store_graph(llm: ChatOpenAI, graph: Neo4jGraph) -> None:
    """Extract entities/relationships from documents and persist to Neo4j."""
    print("  正在用 LLMGraphTransformer 抽取实体和关系…（需要 LLM API 调用）")
    # DeepSeek 不支持 response_format 结构化输出，必须 ignore_tool_usage=True
    transformer = LLMGraphTransformer(llm=llm, ignore_tool_usage=True)
    graph_docs = transformer.convert_to_graph_documents(DOCUMENTS)
    graph.add_graph_documents(graph_docs, baseEntityLabel=True, include_source=True)
    print(f"  ✅ 已将 {len(graph_docs)} 个图文档存入 Neo4j")

    # 展示抽取出的三元组示例
    print("\n  抽取结果示例:")
    for gd in graph_docs[:3]:
        print(f"    节点: {[n.id for n in gd.nodes[:3]]}")
        print(f"    边  : {[(r.source.id, r.type, r.target.id) for r in gd.relationships[:2]]}")


print("=" * 65)
print("STEP 2 — 实体抽取 & 存入 Neo4j")
print("=" * 65)
extract_and_store_graph(llm, graph)

## step 7 — 构建向量检索库 / Build Vector Store

使用 **LangChain + Chroma + MiniLM** 构建向量库：
- 同样的原始文本被向量化
- 用于模糊语义检索（图谱 Cypher 查询的补充）

In [ ]:
def build_vector_store() -> Chroma:
    """Create an in-memory Chroma vector store from the raw documents."""
    embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
    return Chroma.from_texts(texts=RAW_TEXTS, embedding=embeddings)


print("=" * 65)
print("STEP 3 — 构建向量检索库")
print("=" * 65)
vector_store = build_vector_store()
print("✅ Chroma 向量库构建完成（MiniLM embeddings）")

## step 8 — 构建 QA 链 / Build QA Chains

**GraphCypherQAChain** 的内部流程：

```
自然语言问题
    │
    ▼  LLM (Text2Cypher)
Cypher 查询语句
    │
    ▼  Neo4j 执行
图谱查询结果
    │
    ▼  LLM (结果解读)
自然语言答案
```

参数：
- `verbose=False`：关闭调试输出（可改为 `True` 查看生成的 Cypher）
- `return_intermediate_steps=True`：返回中间 Cypher 语句，便于调试
- `allow_dangerous_requests=True`：**langchain-neo4j 安全确认**（LLM 生成的 Cypher 会直接执行；本 demo 使用本地 Neo4j，教学用途可开启）

In [ ]:
def build_graph_chain(llm: ChatOpenAI, graph: Neo4jGraph) -> GraphCypherQAChain:
    """Build a GraphCypherQAChain that translates NL questions to Cypher."""
    return GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=False,
        return_intermediate_steps=True,
        allow_dangerous_requests=True,  # LLM 生成的 Cypher 会直接执行，教学 demo 需显式确认
    )


print("=" * 65)
print("STEP 4 — 构建 QA 链")
print("=" * 65)
graph_chain = build_graph_chain(llm, graph)
print("✅ GraphCypherQAChain 构建完成")

## step 9 — 混合查询函数 / Hybrid Query Function

**两路检索 + LLM 综合**的核心逻辑：

```
问题
 ├── 图谱路径：GraphCypherQAChain → Cypher → 精确结构化答案
 └── 向量路径：similarity_search  → 语义上下文片段
                    │
                    ▼
          LLM 综合 Prompt：
          "图谱答案: ...
           向量上下文: ...
           请结合两者给出最终答案:"
```

In [ ]:
def vector_answer(vector_store: Chroma, question: str, k: int = 3) -> str:
    """Retrieve top-k chunks and concatenate as a simple vector-based answer."""
    docs = vector_store.similarity_search(question, k=k)
    return "\n".join(d.page_content for d in docs)


def hybrid_query(
    question: str,
    graph_chain: GraphCypherQAChain,
    vector_store: Chroma,
    llm: ChatOpenAI,
) -> dict[str, Any]:
    """
    Run the question through both graph and vector pipelines.
    Combine their outputs into a final synthesised answer.
    """
    # 图谱路径
    try:
        graph_result = graph_chain.invoke({"query": question})
        graph_answer = graph_result.get("result", "No graph answer found.")
        cypher_query = (
            graph_result.get("intermediate_steps", [{}])[0]
            .get("query", "N/A")
        )
    except Exception as exc:
        graph_answer = f"Graph chain error: {exc}"
        cypher_query = "N/A"

    # 向量路径
    vector_context = vector_answer(vector_store, question)

    # LLM 综合两路结果
    synthesis_prompt = (
        f"Question: {question}\n\n"
        f"Graph database answer:\n{graph_answer}\n\n"
        f"Vector retrieval context:\n{vector_context}\n\n"
        "Combine both sources and give a concise, accurate final answer:"
    )
    synthesis = llm.invoke([HumanMessage(content=synthesis_prompt)])
    final_answer = synthesis.content

    return {
        "question": question,
        "graph_answer": graph_answer,
        "cypher_query": cypher_query,
        "vector_context_snippet": vector_context[:200] + "…",
        "final_answer": final_answer,
    }


def print_result(result: dict[str, Any]) -> None:
    print(f"\n  Question   : {result['question']}")
    print(f"  Cypher     : {result['cypher_query']}")
    print(f"  Graph Ans  : {result['graph_answer']}")
    print(f"  Vec Context: {result['vector_context_snippet']}")
    print(f"  Final Ans  : {result['final_answer']}")


print("✅ Hybrid Query 函数定义完成")

---
## 🚀 step 10 — 混合查询演示 / Hybrid Query Demo

三个问题覆盖不同场景：

| 问题 | 类型 | 预期最优路径 |
|------|------|-------------|
| Who is the CEO of Apple? | 简单结构化 | 图谱 |
| Which city is Apple's CEO's university located in? | 多跳推理 | 图谱 + 向量 |
| What products does Microsoft make? | 枚举关系 | 图谱 |

In [ ]:
DEMO_QUESTIONS: list[str] = [
    "Who is the CEO of Apple?",
    "Which city is Apple's CEO's university located in?",
    "What products does Microsoft make?",
]

print("=" * 65)
print("STEP 5 — Hybrid Query Demo")
print("=" * 65)

for question in DEMO_QUESTIONS:
    print("-" * 65)
    result = hybrid_query(question, graph_chain, vector_store, llm)
    print_result(result)

---
## 📊 总结 / Takeaway

### 各组件分工

| 组件 | 角色 | 擅长 |
|------|------|------|
| `LLMGraphTransformer` | 实体关系抽取 | 非结构化文本 → 结构化图谱 |
| `Neo4jGraph` | 图存储 + 查询 | 多跳关系遍历，精确结构化 |
| `GraphCypherQAChain` | NL → Cypher → 答案 | 关系型、枚举型问题 |
| `Chroma` + MiniLM | 向量检索 | 语义相似、模糊匹配 |
| LLM 综合 | 答案生成 | 融合两路信息，自然语言输出 |

### 核心洞察

```
图谱链   → 精确关系查询（多跳推理的终极方案）
向量检索 → 语义模糊匹配（知识图谱的补充）
LLM 综合 → 将两路不同质量的信息融合为自然语言答案

GraphRAG = 结构化知识 + 语义检索 + LLM 生成
```

**至此，完整的 GraphRAG 课程演示结束。** 🎉